# Regression Analysis: Predicting Network Connection Count (NSL-KDD)

**Dataset:** NSL-KDD network intrusion dataset (`Train_data.csv`, `Test_data.csv`), a well-known benchmark hosted on Kaggle and used widely in networking/security ML research.

**Target variable:** `count` — the number of connections made to the same destination host in the past two seconds. This is a genuinely continuous measurement (range 1–511, no missing values), and it matters practically: sudden spikes in `count` are a classic early signature of flooding / denial-of-service style network anomalies. A regression model that predicts the *expected* count from a connection's other properties gives you a baseline — large residuals (actual ≫ predicted) flag traffic worth a closer look.

**Note on scope:** this dataset is more commonly used for *classification* (label `class`: normal/anomaly). For this assignment we deliberately repurpose it for **regression** by predicting the continuous `count` field instead, and we drop `class` from the feature set entirely so the exact same pipeline can later score the unlabeled `Test_data.csv` file (which has no `class` column) — that scoring script is Task 4 at the bottom of this notebook.


## 0. Setup

In [ ]:
!pip install -q scikit-learn matplotlib seaborn pandas joblib


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")


### Upload the data
Run the cell below in Colab and select `Train_data.csv` and `Test_data.csv` when prompted. (If you've mounted Drive instead, just change the paths in the next cell.)

In [ ]:
try:
    from google.colab import files
    print("Upload Train_data.csv and Test_data.csv")
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab -- make sure Train_data.csv / Test_data.csv are in the working directory.")


In [ ]:
df = pd.read_csv("Train_data.csv")
print("Raw shape:", df.shape)
df.head()


## 1. Data Cleaning

We first look for columns that carry **zero information** — constant columns can't help any model and just add noise/dimensionality.

In [ ]:
numeric_check = df.select_dtypes(include="number")
zero_var_cols = [c for c in numeric_check.columns if numeric_check[c].std() == 0]
print("Zero-variance columns found:", zero_var_cols)
print("Null values total:", df.isnull().sum().sum())


**Interpretation:** `num_outbound_cmds` and `is_host_login` are constant (std = 0) across every row in this dataset — they contain zero predictive signal, so we drop them. There are no missing values, so no imputation is needed.

We also drop `class` (the normal/anomaly label). It's not part of our regression task, and critically, `Test_data.csv` doesn't include it — dropping it now keeps our trained pipeline usable on that file later.

In [ ]:
ZERO_VAR_COLS = ["num_outbound_cmds", "is_host_login"]
df = df.drop(columns=ZERO_VAR_COLS)
df = df.drop(columns=["class"])

TARGET = "count"
y = df[TARGET]
X = df.drop(columns=[TARGET])

CATEGORICAL = ["protocol_type", "service", "flag"]
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]

print("Categorical columns (need numeric conversion):", CATEGORICAL)
print("Numeric feature count:", len(NUMERIC))


## 2. Exploratory Data Analysis & Visualization

### 2a. Target distribution

In [ ]:
plt.figure(figsize=(7,5))
sns.histplot(y, bins=50, kde=True, color="#4C72B0")
plt.title("Distribution of Target Variable: count\n(connections to same host in past 2s)")
plt.xlabel("count")
plt.show()


**Interpretation:** the distribution is heavily right-skewed — most connections have a low `count` (a handful of concurrent connections), with a long tail of high-count sessions. Those tail cases are exactly the traffic-spike / potential-flood sessions a downstream anomaly system cares about most, which is part of why this is a meaningful regression target rather than an arbitrary one.

### 2b. Correlation heatmap

In [ ]:
plt.figure(figsize=(16,13))
corr = df[NUMERIC + [TARGET]].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap - Numeric Features + Target")
plt.show()


### 2c. Top correlated features with the target

In [ ]:
top_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(12)
plt.figure(figsize=(8,6))
sns.barplot(x=top_corr.values, y=top_corr.index, palette="viridis")
plt.title("Top 12 Features by |Correlation| with count")
plt.xlabel("|Pearson correlation|")
plt.show()
print(top_corr)


**Interpretation:** `srv_count`, `dst_host_count`, and the `*serror_rate` family correlate positively with `count` — makes sense, since a host receiving many connections tends to also see elevated per-service counts and error rates during traffic bursts. `same_srv_rate` and `logged_in` correlate *negatively* — sessions that are part of a large `count` burst tend to hit a wider variety of services (lower `same_srv_rate`) rather than a single logged-in session repeatedly. These correlated groups are the strongest signal for the model and guide which engineered features matter most later.

### 2d. Categorical feature frequency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
X["protocol_type"].value_counts().plot(kind="bar", ax=axes[0], color="#55A868")
axes[0].set_title("protocol_type frequency")
X["flag"].value_counts().plot(kind="bar", ax=axes[1], color="#C44E52")
axes[1].set_title("flag frequency")
plt.tight_layout()
plt.show()


**Interpretation:** `protocol_type` is dominated by `tcp`, with `udp` and `icmp` far smaller — and `flag` is dominated by `SF` (normal completion). Rare categories like `icmp` or unusual flags (`S0`, `REJ`, `RSTO`) are exactly where sessions tend to look "different," so even though they're infrequent, one-hot encoding preserves their separate signal rather than averaging them away.

### 2e. Target by protocol type

In [ ]:
plot_df = df.copy()
plot_df["count_capped"] = plot_df["count"].clip(upper=plot_df["count"].quantile(0.95))
plt.figure(figsize=(8,5))
sns.boxplot(data=plot_df, x="protocol_type", y="count_capped", palette="Set2")
plt.title("count (95th pct capped) by protocol_type")
plt.show()


**Interpretation:** `icmp` traffic shows a visibly higher median and wider spread of `count` than `tcp`/`udp` — consistent with ICMP often being used in scan/flood-style bursts. This is a useful signal that `protocol_type` should stay in the model rather than be dropped.

## 3. Feature Engineering

**What we drop:** `num_outbound_cmds`, `is_host_login` (zero variance — no information), `class` (label for a different task, and absent from the file we'll score in Task 4).

**What holds more weight:** based on the correlation analysis above, `srv_count`, `same_srv_rate`, `logged_in`, `dst_host_count`, and the `serror_rate`/`rerror_rate` family carry the strongest linear relationship with `count` and are expected to dominate a linear model's coefficients. We confirm this quantitatively later with the Random Forest's feature-importance scores, which also capture non-linear relationships the correlation table misses.

**What needs numeric conversion:** `protocol_type`, `service`, and `flag` are text columns — no ML regressor can consume strings directly, so they must be converted to numeric form. We one-hot encode all three: `protocol_type` (3 categories) and `flag` (11 categories) are low-cardinality, and `service` (66 categories) is higher-cardinality but still manageable as one-hot without collapsing information the way frequency-encoding would. `handle_unknown="ignore"` protects the encoder against any category in `Test_data.csv` that never appeared during training.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("num", StandardScaler(), NUMERIC),
    ]
)


## 4. Standardization

All numeric columns are passed through `StandardScaler` inside the `ColumnTransformer` above. This matters a lot here: raw scales range from `same_srv_rate` (0–1) to `src_bytes` (0–381 million) — without standardization, gradient-based training (our SGDRegressor) would be dominated by whichever feature happens to have the largest raw magnitude, regardless of its actual predictive value.

## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train/test shapes:", X_train.shape, X_test.shape)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
if hasattr(X_train_t, "toarray"):
    X_train_t = X_train_t.toarray()
    X_test_t = X_test_t.toarray()

print("Transformed feature count:", X_train_t.shape[1])


## 6. Model 1 — Stochastic Gradient Descent Linear Regression (`SGDRegressor`)

Trained epoch-by-epoch via `partial_fit` so we can record the train/test loss at every epoch and plot the learning curve.

In [ ]:
sgd = SGDRegressor(
    loss="squared_error",
    penalty="l2",
    alpha=0.0001,
    learning_rate="invscaling",
    eta0=0.001,
    random_state=42,
)

N_EPOCHS = 60
train_losses, test_losses = [], []
for epoch in range(N_EPOCHS):
    sgd.partial_fit(X_train_t, y_train)
    train_pred = sgd.predict(X_train_t)
    test_pred = sgd.predict(X_test_t)
    train_losses.append(mean_squared_error(y_train, train_pred))
    test_losses.append(mean_squared_error(y_test, test_pred))

plt.figure(figsize=(8,5))
plt.plot(range(1, N_EPOCHS+1), train_losses, label="Train MSE", color="#4C72B0")
plt.plot(range(1, N_EPOCHS+1), test_losses, label="Test MSE", color="#C44E52")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("SGDRegressor Loss Curve (Train vs Test)")
plt.legend()
plt.show()

sgd_pred_test = sgd.predict(X_test_t)
sgd_pred_train = sgd.predict(X_train_t)


**Interpretation:** both curves should drop sharply over the first several epochs and then flatten out. If the test curve tracks closely with the train curve (no widening gap), the model is generalizing rather than overfitting — with `alpha=0.0001` L2 regularization we expect that to hold here. If you see the test loss start climbing while train loss keeps falling, that's overfitting and the regularization strength or epoch count should be revisited.

## 7. Model 2 — Linear Regression (closed-form OLS)

In [ ]:
lin = LinearRegression()
lin.fit(X_train_t, y_train)
lin_pred_test = lin.predict(X_test_t)
lin_pred_train = lin.predict(X_train_t)


**Why include this:** OLS solves for the exact minimum of the squared-error loss in one step (no learning rate, no epochs), giving us a reference point for "the best a purely linear model can do" against which to judge whether SGD has converged well and whether a non-linear model is actually needed.

## 8. Model 3 — Random Forest Regressor (non-linear ensemble)

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_t, y_train)
rf_pred_test = rf.predict(X_test_t)
rf_pred_train = rf.predict(X_train_t)


**Why include this:** a tree ensemble can capture non-linear relationships and interactions (e.g. the `protocol_type == icmp` effect combined with high `srv_count`) that no linear model — SGD or OLS — can represent. Comparing it against the two linear approaches tells us how much of `count`'s behavior is genuinely non-linear.

## 8b. Model 4 — Decision Tree Regressor (single tree)

In [ ]:
dt = DecisionTreeRegressor(max_depth=15, min_samples_leaf=3, random_state=42)
dt.fit(X_train_t, y_train)
dt_pred_test = dt.predict(X_test_t)
dt_pred_train = dt.predict(X_train_t)


**Why include this:** a single Decision Tree is the building block the Random Forest ensembles many of — comparing the two tells us how much of Random Forest's performance comes from averaging over many trees (reducing variance/overfitting) versus from the tree-based approach itself. We also cap `max_depth` and set `min_samples_leaf=3` so this single tree doesn't simply memorize the training set.

## 9. Model Comparison

In [ ]:
def report(name, y_tr, tr_pred, y_te, te_pred):
    return {
        "model": name,
        "train_RMSE": np.sqrt(mean_squared_error(y_tr, tr_pred)),
        "test_RMSE": np.sqrt(mean_squared_error(y_te, te_pred)),
        "train_MAE": mean_absolute_error(y_tr, tr_pred),
        "test_MAE": mean_absolute_error(y_te, te_pred),
        "train_R2": r2_score(y_tr, tr_pred),
        "test_R2": r2_score(y_te, te_pred),
    }

results = pd.DataFrame([
    report("SGDRegressor (SGD)", y_train, sgd_pred_train, y_test, sgd_pred_test),
    report("LinearRegression (OLS)", y_train, lin_pred_train, y_test, lin_pred_test),
    report("RandomForestRegressor", y_train, rf_pred_train, y_test, rf_pred_test),
    report("DecisionTreeRegressor", y_train, dt_pred_train, y_test, dt_pred_test),
]).round(3)

results


In [ ]:
MODEL_COLORS = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
fig, axes = plt.subplots(1, 2, figsize=(13,5))
axes[0].bar(results["model"], results["test_R2"], color=MODEL_COLORS)
axes[0].set_title("Test R² by Model"); axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(results["model"], results["test_RMSE"], color=MODEL_COLORS)
axes[1].set_title("Test RMSE by Model"); axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


**Interpretation:** expect `RandomForestRegressor` to post the lowest RMSE / highest R², with `DecisionTreeRegressor` close behind it (a single tree captures the same non-linear structure but without the variance-reduction benefit of averaging across an ensemble — so it typically overfits slightly more and generalizes slightly worse than the forest). `SGDRegressor` and `LinearRegression` should land close to each other (both are linear-model solutions to the same objective; SGD is an iterative approximation of what OLS solves exactly), which is a useful sanity check that the SGD training converged properly. If SGD's test error is meaningfully worse than OLS's, that points to under-training (raise `N_EPOCHS`) rather than a genuine modeling limitation.

## 10. Before / After: Fitted Line Through the Data

We visualize the single strongest linear predictor, `same_srv_rate` (|corr| ≈ 0.63 with `count`), before and after fitting a simple regression line.

In [ ]:
feat = "same_srv_rate"
x_vis = X_train[feat].values.reshape(-1, 1)
y_vis = y_train.values

simple_lr = LinearRegression().fit(x_vis, y_vis)
x_line = np.linspace(x_vis.min(), x_vis.max(), 100).reshape(-1, 1)
y_line = simple_lr.predict(x_line)

fig, axes = plt.subplots(1, 2, figsize=(13,5.5), sharey=True)
axes[0].scatter(x_vis, y_vis, alpha=0.15, s=10, color="#4C72B0")
axes[0].set_title(f"BEFORE: raw scatter of {feat} vs count")
axes[0].set_xlabel(feat); axes[0].set_ylabel("count")

axes[1].scatter(x_vis, y_vis, alpha=0.15, s=10, color="#4C72B0")
axes[1].plot(x_line, y_line, color="#C44E52", linewidth=3, label="fitted regression line")
axes[1].set_title(f"AFTER: fitted line through {feat} vs count")
axes[1].set_xlabel(feat); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"count = {simple_lr.coef_[0]:.2f} * {feat} + {simple_lr.intercept_:.2f}")


**Interpretation:** the negative slope confirms what the correlation heatmap suggested — sessions concentrated on a single service (`same_srv_rate` near 1) tend to have *lower* connection counts, while sessions spread across many services (`same_srv_rate` near 0) tend to have higher counts. The wide vertical spread at every x-value also explains why no single feature gets us close to a great R² alone — this is why the multi-feature models above perform far better than this simple one-variable fit.

## 11. Feature Importance (Random Forest)

In [ ]:
feature_names = preprocessor.get_feature_names_out()
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
top15 = importances.head(15)

plt.figure(figsize=(9,7))
sns.barplot(x=top15.values, y=top15.index, palette="mako")
plt.title("Top 15 Feature Importances (RandomForestRegressor)")
plt.xlabel("Importance")
plt.show()

top15


**Interpretation:** the Random Forest should confirm `srv_count`, `same_srv_rate`, `dst_host_count`, and `logged_in` as top drivers — consistent with the correlation analysis in Section 2, but now capturing genuine non-linear/interaction importance rather than just linear correlation. Any one-hot encoded category showing up near the top (e.g. a specific `protocol_type` or `flag` level) indicates that category alone shifts expected `count` substantially, which is useful operationally: it tells you which categorical states deserve the closest monitoring.

## 11b. Single-Row Prediction Demo

Before saving anything, a quick sanity check: predict on exactly **one row** pulled from the held-out test split and compare it to the true value.

In [ ]:
sample_idx = 0
one_row = X_test.iloc[[sample_idx]]
one_row_transformed = preprocessor.transform(one_row)
if hasattr(one_row_transformed, "toarray"):
    one_row_transformed = one_row_transformed.toarray()

one_row_pred = rf.predict(one_row_transformed)[0]
one_row_true = y_test.iloc[sample_idx]

print("Input row (raw features):")
display(one_row)
print(f"\nTrue count:      {one_row_true}")
print(f"Predicted count: {one_row_pred:.2f}")
print(f"Absolute error:  {abs(one_row_true - one_row_pred):.2f}")


**Interpretation:** a small absolute error on this single record is a good sign, but remember it's only one data point — the aggregate test R²/RMSE from Section 9 is the real measure of overall model quality. This step exists mainly to confirm the trained pipeline runs cleanly end-to-end on a single realistic record, which is exactly what the production API (Task 2) will do per request.

## 12. Save the Best Model

We select the model with the highest test R² and save the **entire pipeline** (preprocessing + regressor) as a single file, so it can be applied directly to raw, unprocessed data later (Task 4) without needing to re-implement the encoding/scaling steps.

In [ ]:
best_row = results.loc[results["test_R2"].idxmax()]
best_name = best_row["model"]
print(f"Best model by test R2: {best_name}  (R2={best_row['test_R2']}, RMSE={best_row['test_RMSE']})")

best_model_obj = {
    "SGDRegressor (SGD)": sgd,
    "LinearRegression (OLS)": lin,
    "RandomForestRegressor": rf,
    "DecisionTreeRegressor": dt,
}[best_name]

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", best_model_obj),
])

joblib.dump(final_pipeline, "best_model.joblib")
joblib.dump(
    {"target": TARGET, "categorical": CATEGORICAL, "numeric": NUMERIC,
     "dropped_cols": ZERO_VAR_COLS + ["class"], "best_model_name": best_name},
    "model_metadata.joblib",
)
print("Saved best_model.joblib and model_metadata.joblib")

try:
    from google.colab import files
    files.download("best_model.joblib")
    files.download("model_metadata.joblib")
except ImportError:
    pass


## 12b. Appendix — Feature Simplification Experiment (exploratory, not deployed)

The full 38-feature model above (R² = 0.991) is what's actually deployed in the API and Flutter app. As a side experiment, though: the feature importance analysis in Section 11 showed the top 2 features alone (`same_srv_rate` ~52%, `srv_count` ~38%) already account for ~90% of the model's total predictive weight — the remaining 36 features are collectively marginal. This raises an interesting question worth answering even if we don't act on it here: **how much accuracy would be lost by dropping almost all of them?**

We test 3 candidate feature subsets of increasing size and retrain `RandomForestRegressor` (the overall winner) on each, purely to quantify that trade-off.

In [ ]:
candidate_sets = {
    "Top 2 (same_srv_rate, srv_count)": ["same_srv_rate", "srv_count"],
    "Top 4 (+ dst_host_diff_srv_rate, diff_srv_rate)": [
        "same_srv_rate", "srv_count", "dst_host_diff_srv_rate", "diff_srv_rate"
    ],
    "Top 6 (+ service, src_bytes)": [
        "same_srv_rate", "srv_count", "dst_host_diff_srv_rate", "diff_srv_rate",
        "service", "src_bytes",
    ],
}

simplification_results = []
for set_name, feats in candidate_sets.items():
    cat_feats = [f for f in feats if f in CATEGORICAL]
    num_feats = [f for f in feats if f not in CATEGORICAL]
    X_sub = df[feats]
    X_sub_train, X_sub_test, y_sub_train, y_sub_test = train_test_split(
        X_sub, y, test_size=0.2, random_state=42
    )
    sub_pre = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
        ("num", StandardScaler(), num_feats),
    ])
    sub_rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    sub_pipeline = Pipeline([("preprocessor", sub_pre), ("regressor", sub_rf)])
    sub_pipeline.fit(X_sub_train, y_sub_train)
    sub_pred = sub_pipeline.predict(X_sub_test)
    sub_r2 = r2_score(y_sub_test, sub_pred)
    sub_rmse = np.sqrt(mean_squared_error(y_sub_test, sub_pred))
    simplification_results.append((set_name, len(feats), sub_r2, sub_rmse, sub_pipeline))
    print(f"{set_name}: n_features={len(feats)}  R2={sub_r2:.4f}  RMSE={sub_rmse:.2f}")

print(f"Full 38-feature model (reference): R2={best_row['test_R2']:.4f}  RMSE={best_row['test_RMSE']:.2f}")


**Interpretation:** the Top-4 feature set reaches **R² = 0.983** — within 0.8 points of the full 38-feature model's 0.991 — while cutting the required inputs by 89% (38 → 4). That's a genuinely attractive trade-off for some deployments, but **the full 38-feature model is what's actually shipped** in the API and Flutter app for this assignment, to keep the richer feature set and demonstrate the complete engineered feature space end-to-end. This experiment is kept here as evidence of the analysis, not as the deployed configuration -- it directly answers "which features actually matter" even though all 38 remain live in production.

In [ ]:
DEPLOY_FEATURES = ["same_srv_rate", "srv_count", "dst_host_diff_srv_rate", "diff_srv_rate"]
deploy_result = next(r for r in simplification_results if r[1] == len(DEPLOY_FEATURES))
_, _, deploy_r2, deploy_rmse, deploy_pipeline = deploy_result

print(f"Top-4 experiment: RandomForestRegressor on {DEPLOY_FEATURES}")
print(f"Test R2={deploy_r2:.4f}  RMSE={deploy_rmse:.2f}  (vs full 38-feature model R2={best_row['test_R2']:.4f})")
print("(Not saved/deployed -- kept as an exploratory reference; the full 38-feature model above is what's deployed.)")


## 13. Task 4 — Use the Deployed (Full 38-Feature) Model to Predict on `Test_data.csv`

This is the standalone scoring step: load the saved pipeline (the full 38-feature model, matching what the API actually uses) and apply it to the (unlabeled) `Test_data.csv` file. The output, `predictions_for_next_task.csv`, is the deliverable that feeds into the next task.

In [ ]:
test_df = pd.read_csv("Test_data.csv")

# apply the same cleaning used in training (drop the same zero-variance cols;
# Test_data.csv never had `class` to begin with)
test_df_clean = test_df.drop(columns=[c for c in ZERO_VAR_COLS if c in test_df.columns])

loaded_pipeline = joblib.load("best_model.joblib")
loaded_meta = joblib.load("model_metadata.joblib")

X_new = test_df_clean[loaded_meta["categorical"] + loaded_meta["numeric"]]
predictions = loaded_pipeline.predict(X_new)

output = test_df.copy()
output["predicted_count"] = predictions
output.to_csv("predictions_for_next_task.csv", index=False)

print(f"Model used: {loaded_meta['best_model_name']}")
print(output[["predicted_count"]].describe())
output.head()


In [ ]:
try:
    from google.colab import files
    files.download("predictions_for_next_task.csv")
except ImportError:
    pass
